In [ ]:
#!pip install --upgrade pip setuptools wheel
#!pip install -r ./../requirements.txt 

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------  8.9/9.1 MB 102.6 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 36.9 MB/s  0:00:00
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------- ----------------- 22.8/40.2 MB 112.7 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 104.2 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 104.2 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 104.2 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 46.9 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------  6.8/6.9 MB 86.5 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 29.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [opencv-python]
   -------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

# Lecture des données
X_train = pd.read_csv("../data/X_train_update.csv")
y_train = pd.read_csv("../data/Y_train_CVw08PX.csv")
X_test = pd.read_csv("../data/X_test_update.csv")

# Affichage des informations sur les datasets
print(f"Info X_train : {X_train.info()}")
print(f"Info Y_train : {y_train.info()}")
print(f"Info X_test : {X_test.info()}")

# Affichage des tailles des datasets
print(f"Taille X_train : {X_train.shape}")
print(f"Taille Y_train : {y_train.shape}")
print(f"Taille X_test : {X_test.shape}")

# Affichage du nombre de classes dans la variable cible
print(y_train['prdtypecode'].nunique())  # 27 classes

# Merge données d'entrainement
full_data = pd.merge(X_train, y_train, left_index=True, right_index=True)

# Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

# Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)
full_data.set_index(['id'], inplace=True)

X_test.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
X_test.set_index(['id'], inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   84916 non-null  int64 
 1   designation  84916 non-null  object
 2   description  55116 non-null  object
 3   productid    84916 non-null  int64 
 4   imageid      84916 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 3.2+ MB
Info X_train : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   84916 non-null  int64
 1   prdtypecode  84916 non-null  int64
dtypes: int64(2)
memory usage: 1.3 MB
Info Y_train : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13812 entries, 0 to 13811
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1

In [ ]:
# Merge données d'entrainement
full_data = pd.merge(X_train, y_train, left_index=True, right_index=True)

# Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

# Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)
full_data.set_index(['id'], inplace=True)

X_test.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
X_test.set_index(['id'], inplace=True)

In [ ]:
# Fonction de nettoyage du texte
import re

def clean_text(text):
    if pd.isnull(text):
        return ""

    # Supprimer la ponctuation
    #text = re.sub(r'[^\w\s]', '', text)

    # Supprimer balises HTML
    text = re.sub(r'<.*?>', '', text)

    # Remplacer <br /> par un espace
    text = text.replace(r'<br />', ' ')

    # Remplacer les référence de caractère HTML
    text = text.replace(r'&amp;', '&')
    text = text.replace(r'&nbsp;', ' ')
    text = text.replace(r'&lt', '<')
    text = text.replace(r'&gt', '>')
    text = text.replace(r'&quot', '"')
    text = text.replace(r'&#39', "'")
    text = text.replace(r'&eacute', 'e')
    text = text.replace(r'&egrave', 'e')
    text = text.replace(r'&ecirc', 'e')
    
    # Convertir en minuscules
    text = text.lower()

    # Supprimer les espaces supplémentaires
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Nettoyage du texte
full_data['clean_designation'] = full_data['designation'].apply(clean_text)
full_data['clean_description'] = full_data['description'].apply(clean_text)

# Vérification de doublon dans les désignations et descriptions nettoyées
duplicate_designation = full_data['clean_designation'].duplicated().sum()
duplicate_description = full_data['clean_description'].duplicated().sum()

# Concaténation des désignations et descriptions nettoyées pour l'analyse de texte
full_data['text'] = full_data['clean_designation'] + ' ' + full_data['clean_description']

# Création d'un nouveau dataframe pour l'analyse de texte
test_data = full_data.drop(columns=['designation', 'description', 'productid', 'imageid','clean_designation','clean_description', 'len_description', 'len_designation', 'nb_words_description', 'nb_words_designation'], axis=1)
display(test_data.head(20))

KeyError: "['len_description', 'len_designation', 'nb_words_description', 'nb_words_designation'] not found in axis"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# Importation de TfidfVectorizer pour la vectorisation du texte
vectorizer = TfidfVectorizer()

# Création des ensembles d'entraînement et de test pour l'analyse de texte (20% pour le test)
X_train, X_test, y_train, y_test = train_test_split(full_data['text'], full_data['prdtypecode'], test_size=0.2, random_state=42)

# Vectorisation du texte
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
# Test de modèle KNN sur les données vectorisées
# Création du classifieur KNN
clf = KNeighborsClassifier(n_neighbors=5)

# Entraînement du classifieur
clf.fit(X_train, y_train)

# Prédiction sur l'ensemble de test
y_pred = clf.predict(X_test)

# Évaluation de la précision du classifieur
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du classifieur KNN : {accuracy:.4f}")

# Matrice de confusion
cm = pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted'])
print("Matrice de confusion :")
print(cm)

In [ ]:
# Test de différents k et différentes métriques pour le KNN
from sklearn import neighbors

score_minko = []
score_man = []
score_cheb = []

for k in range (1,11):
    knn_minko = neighbors.KNeighborsClassifier(n_neighbors=k, metric='minkowski')
    knn_minko.fit(X_train, y_train)
    score_minko.append(knn_minko.score(X_test, y_test))

for k in range (1,11):
    knn_man = neighbors.KNeighborsClassifier(n_neighbors=k, metric='manhattan')
    knn_man.fit(X_train, y_train)
    score_man.append(knn_man.score(X_test, y_test))

for k in range (1,11):
    knn_cheb = neighbors.KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_cheb.fit(X_train, y_train)
    score_cheb.append(knn_cheb.score(X_test, y_test))

In [ ]:
# Affichage dans un graphique les scores des différents métriques en fonction de K
plt.plot(range(1,11), score_minko, color='red', linestyle='--', label='Mink')
plt.plot(range(1,11), score_man, color='blue', linestyle='-',lw=0.5, label='Man')
plt.plot(range(1,11), score_cheb, color='green', linestyle=':', lw=3, label='Cheb') 

plt.legend()
plt.title('Score / valeur de K')
plt.xlabel('Valeur de K')
plt.ylabel('Accuracy')
plt.grid()
plt.show()